# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [46]:
!pip install gdown

In [47]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone
from sklearn.metrics import accuracy_score
import joblib
import gdown

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [48]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, timestamp_col='timestamp'):
        self.timestamp_col = timestamp_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df[self.timestamp_col] = pd.to_datetime(df[self.timestamp_col])
        df['hour'] = df[self.timestamp_col].dt.hour
        df['dayofweek'] = df[self.timestamp_col].dt.weekday
        df['target'] = df['dayofweek'] < 5
        df = df.drop(columns=[self.timestamp_col])
        return df

In [49]:
from sklearn.base import BaseEstimator, TransformerMixin

class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col=None, drop_first=True):
        self.target_col = target_col
        self.drop_first = drop_first
        self.categorical_columns_ = None

    def fit(self, X, y=None):
        cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.target_col and self.target_col in cols:
            cols.remove(self.target_col)
        self.categorical_columns_ = cols
        return self

    def transform(self, X):
        df = X.copy()
        if self.categorical_columns_:
            df = pd.get_dummies(df, columns=self.categorical_columns_, drop_first=self.drop_first)
        return df

In [50]:
class TrainValidationTest:
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def split(self):
        X_train, X_temp, y_train, y_temp = train_test_split(self.X, self.y, test_size=0.4, random_state=21, stratify=self.y)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=21, stratify=y_temp)
        
        return X_train, X_val, X_test, y_train, y_val, y_test
    

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [62]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results_ = []

    def choose(self, X_train, y_train, X_valid, y_valid):
        self.results_ = []

        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f"\nEstimator: {model_name}")

            # Fit GridSearchCV with progress bar
            with tqdm(total=1, desc=model_name):
                grid.fit(X_train, y_train)

            best_estimator = grid.best_estimator_
            best_params = grid.best_params_

            # Training accuracy
            train_acc = accuracy_score(
                y_train, best_estimator.predict(X_train)
            )

            # Validation accuracy
            valid_acc = accuracy_score(
                y_valid, best_estimator.predict(X_valid)
            )

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {train_acc:.3f}")
            print(f"Validation set accuracy score for best params: {valid_acc:.3f}")

            self.results_.append({
                "model": model_name,
                "params": best_params,
                "valid_score": valid_acc,
                "estimator": best_estimator
            })

        best_model = max(self.results_, key=lambda x: x["valid_score"])
        print(f"\nClassifier with best validation set accuracy: {best_model['model']}")

        return best_model["model"]


    def best_results(self):
        return pd.DataFrame(
            [{
                "model": r["model"],
                "params": r["params"],
                "valid_score": r["valid_score"]
            } for r in self.results_]
        )


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [52]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {acc}")

        return acc

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model was successfully saved to {path}")


## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [53]:
url='https://drive.google.com/uc?id=14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw&export=download'

output='checker-submits.csv'
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw&export=download
To: /home/mirshod/Desktop/DSB11_ML_Advanced.ID_886524-1/src/ex04/checker-submits.csv
100%|██████████| 77.0k/77.0k [00:00<00:00, 182kB/s]


'checker-submits.csv'

In [54]:
df=pd.read_csv('checker-submits.csv')
df.head()

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104


In [55]:
preprocessing = Pipeline(steps=[
    ('feature_extraction', FeatureExtractor()),
    ('one_hot_encoding', MyOneHotEncoder(target_col='target'))
])
df = preprocessing.fit_transform(df)
df.head()

,numTrials,hour,dayofweek,target,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,4,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,2,5,4,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,3,5,4,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,4,5,4,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,5,5,4,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [56]:
y = df['target']
X = df.drop(columns=['target'])
X_train, X_val, X_test, y_train, y_val, y_test = TrainValidationTest(X, y).split()

In [57]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [63]:
svm_params={
    'kernel':['linear', 'rbf', 'sigmoid'],
    'C':[0.01, 0.1, 1, 1.5, 5, 10],
    'gamma':['scale', 'auto'],
    'class_weight':['balanced', None],
    'random_state':[21],
    'probability':[True],
}

dtree_params={
    'max_depth':range(1, 50),
    'class_weight':['balanced', None],
    'criterion':['entropy', 'gini'],
}

random_forest_params={
    'n_estimators':[5, 10, 50, 100],
    'max_depth':range(1, 50),
    'class_weight':['balanced', None],
    'criterion':['entropy', 'gini']
}


dtree_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=21),
    param_grid=dtree_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

svm_grid = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

random_forest_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=21),
    param_grid=random_forest_params,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grids = [dtree_grid, svm_grid, random_forest_grid]

grid_dict = {
    0: "Decision Tree",
    1: "SVM",
    2: "Random Forest"
}

model_selector = ModelSelection(grids, grid_dict)

best_model_name = model_selector.choose(
    X_train, y_train,
    X_val, y_val
)

results_df = model_selector.best_results()
results_df


Estimator: Decision Tree


Decision Tree:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 1}
Best training accuracy: 1.000
Validation set accuracy score for best params: 1.000

Estimator: SVM


SVM:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'C': 0.1, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'linear', 'probability': True, 'random_state': 21}
Best training accuracy: 1.000
Validation set accuracy score for best params: 1.000

Estimator: Random Forest


Random Forest:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 2, 'n_estimators': 100}
Best training accuracy: 1.000
Validation set accuracy score for best params: 1.000

Classifier with best validation set accuracy: Decision Tree


,model,params,valid_score
0,Decision Tree,"{'class_weight': 'balanced', 'criterion': 'ent...",1.0
1,SVM,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'...",1.0
2,Random Forest,"{'class_weight': 'balanced', 'criterion': 'ent...",1.0


`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [64]:
rand_forest_best_params=[
    r['params'] for r in model_selector.results_ if r['model'] == 'Random Forest'
][0]

best_estimator=RandomForestClassifier(**rand_forest_best_params, random_state=21)
final=Finalize(
    estimator=best_estimator
)

final_score = final.final_score(X_train, y_train, X_test, y_test)
final_score

Accuracy of the final model is 1.0


1.0

In [65]:
final.save_model('final_model.joblib')

Model was successfully saved to final_model.joblib
